ML 因子 notebook (xgb 推理: booster 优先, 任意异常自动回退纯 numpy 树遍历, 平台版本无关)。

In [ ]:
# -*- coding: utf-8 -*-
"""R57b 路B ML 因子 (ml_xgb_wf35_pathB) 平台推理管线 — 多文件包成员。
35 特征平台侧现算 (权威实现对应 scripts/mine_r57b_mltrain.py 的 FEATURES 表,
公式逐一对齐各批次挖掘脚本/提交 notebook, 缓存因子为带符号值的已按要求乘 sign):
  L1 deal_number 系 (R45/R46/R52 notebook 口径: bar1m volume/deal_number/amount
     当日累计 -> 组内差分; close/盘口快照不差分);
  L2 在榜基底 4 (R38/R37/R20 notebook 公式, 含 MAD-winsorize 截面 z 与 tilt);
  L3 波动族 (R47/R52/R54/R55 挖掘口径 + factorlib daily_return 系 + 1d Parkinson/GK);
  L4 历史强因子 (R17-R20 notebook 公式; 注意两套组内 corr 约定: R45 系 min_n->NaN,
     R17/R20 系 n<30 或零方差 -> 0.0)。
模型: XGBoost walk-forward 年度 fold (models/model_YYYY.json); 预测年 Y 用
  model_Y (Y<=2020->2020, 2021->2021, 2022->2022, 2023->2023, >=2024->2024)。
  优先 xgboost.Booster, 无 xgboost 时纯 numpy 向量化树遍历兜底 (79 号 notebook 同思路,
  已向量化, 无 Python 行循环)。
输出: 逐日截面 plain z 后的特征面板 fillna(0) -> 模型 -> SIGN=+1 ->
  bigalpha_2026_instruments inner merge。特征 NaN 处理与训练严格一致
  (inner join 35 特征 -> 逐日 z (std=0 -> NaN) -> fillna(0))。
"""
import json
import os

import numpy as np
import pandas as pd

SIGN = 1

FEATURE_COLS = [
    "deals_absret_corr", "deals_ac1", "bigdeal_ratio", "updn_asym_15m",
    "updn_asym_ma5", "ret_tail3", "resid_rv5_20", "z_retrange30_volac1",
    "retmax15_cashq", "retmax15_cffps", "vwapdevchg5_cffps", "gk5", "park5",
    "vol_ts_5_20", "rv5_tsz20", "vwap_disp", "imb_x_ret_std", "exec_int_mean",
    "exec_int_std", "vollead_corr_ma5", "turn", "rv_skew_ma5", "rv_skew_ma5_15m",
    "upvol_asym_ma5", "kurt_ma5", "n_reversals_30m", "ret_max_15m",
    "absret_ac1_ma5", "vwap_dev_chg5", "gap_freq_20", "b_pvsign_resid_rank_top1",
    "down_vol_share_15", "nm_vol_ret_corr", "nm_n_reversals", "upvol_asym_ma5_15m",
    # R62 扩展 48
    "depth_mean", "depth_std", "depth_skew", "book_slope_mean", "book_slope_std",
    "slope_asym_std", "osize_asym_mean", "osize_asym_std", "imb3_std", "imb1_std",
    "imb1_skew", "imb_x_ret_mean", "mp_dev_mean", "mp_dev_std", "spread_mean",
    "spread_std", "imb3_ac1", "mp_dev_ac1", "orders_ret_corr", "osize_ret_corr",
    "spread_am", "imb3_am", "depth_am",
    "vollead_corr", "deallead_corr", "retlead_corr", "absretlead_corr",
    "mkt_corr", "mkt_corr_am", "amt_center", "deal_center",
    "deals_total", "deals_mean", "deals_std", "deals_skew", "deals_cv",
    "deals_updn_asym",
    "fill_asym_mean", "exec_x_ret", "deals_orders_corr",
    "deals_tail1_share", "vol_tail1_share", "ret_tail1", "tail1_deal_size",
    "tail3_deals_share",
    "vol_top_third", "vol_bot_third", "vwap_skew",
]

MORNING = (575, 630)  # 09:35-10:30

# ---------------------------------------------------------------- 数据访问

def _iter_month_ranges(q_start, q_end):
    q_start = pd.Timestamp(q_start)
    q_end = pd.Timestamp(q_end)
    cur = pd.Timestamp(year=q_start.year, month=q_start.month, day=1)
    while cur <= q_end:
        nxt = cur + pd.offsets.MonthBegin(1)
        cs = max(q_start, cur)
        ce = min(q_end, nxt - pd.Timedelta(seconds=1))
        if cs <= ce:
            yield cs, ce
        cur = nxt


def _dai_query_monthly(sql, q_start, q_end):
    """按月分片查询, 避免一次性加载全量 bar1m 导致 OOM。"""
    import dai
    frames = []
    for cs, ce in _iter_month_ranges(q_start, q_end):
        part = dai.query(
            sql,
            filters={"date": [cs.strftime("%Y-%m-%d %H:%M:%S"),
                              ce.strftime("%Y-%m-%d %H:%M:%S")]},
            compression=True,
        ).df()
        if len(part):
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _load_bar1m(datasources, start_date, end_date, buf_days=100):
    """平台 bar1m: volume/deal_number/amount 当日累计 -> 组内差分 (R45 结论);
    close/open/盘口字段为快照, 不差分 (R47 结论)。"""
    bar1m = datasources["bar1m"]
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=buf_days)
    q_end = pd.to_datetime(end_date)
    sql = (f"SELECT date, instrument::STRING AS instrument, open, high, low, close, "
           f"pre_close, volume, amount, deal_number, bid_price1, ask_price1, "
           f"bid_price2, bid_price3, ask_price2, ask_price3, "
           f"bid_volume1, bid_volume2, bid_volume3, "
           f"ask_volume1, ask_volume2, ask_volume3, "
           f"bid_num_orders1, bid_num_orders2, bid_num_orders3, "
           f"ask_num_orders1, ask_num_orders2, ask_num_orders3 "
           f"FROM {bar1m} WHERE close > 0 ORDER BY instrument, date")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["td"] = df["date"].dt.normalize()
    df["mins"] = df["date"].dt.hour * 60 + df["date"].dt.minute
    gk = ["instrument", "td"]
    for col, out in (("volume", "vol_min"), ("amount", "amt_min"),
                     ("deal_number", "dn_min")):
        df[out] = df[col] - df.groupby(gk, sort=False)[col].shift(1)
        df[out] = df[out].fillna(df[col]).clip(lower=0)
    return df


def _load_factorlib(start_date, end_date, buf_days=100):
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=buf_days)
    q_end = pd.to_datetime(end_date)
    sql = "SELECT date, instrument, daily_return, turn FROM bigalpha_2026_factorlib"
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    return df.sort_values(["instrument", "date"]).reset_index(drop=True)


def _fin_grid_end(end_date):
    """财务 ffill 的 canonical 窗口右端 (R58 移植发现, 见 _load_fin 注释):
    缓存构建用 grid 2019-01-01..2024-12-31 + fin 2018-07..2024-12-31;
    verify_f8 证明右端取 2023-12-31 即可逐值复现缓存 (≤2023-12-31),
    生产期 (>2023-12-31) 用实际 end_date 向右自然延展。"""
    return max(pd.to_datetime(end_date), pd.Timestamp("2023-12-31"))


def _load_fin(datasources, start_date, end_date):
    """财务 PIT, 固定 canonical 窗口 2018-07-01 .. _fin_grid_end(end_date)。
    注意 (R58 移植发现): mine_r26_tilt2.fin_components 的 grid.merge+ffill 结果
    依赖合并后行序 (当前 pandas 下左合并不保序, 乱序 ffill 成为缓存构建的一部分),
    只有用与缓存构建相同的 canonical 输入窗口才能复现缓存值。"""
    fin = datasources["financial"]
    q_start = pd.Timestamp("2018-07-01")
    q_end = _fin_grid_end(end_date)
    sql = (f"SELECT date, instrument::STRING AS instrument, category, "
           f"net_cffoa, net_profit, net_cfffa, latest_shares "
           f"FROM {fin} WHERE shift=0")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    return df


def _stk(start_date, end_date):
    import dai
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [str(pd.to_datetime(start_date)),
                                      str(pd.to_datetime(end_date))]}).df()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    stk["instrument"] = stk["instrument"].astype(str)
    return stk


# ---------------------------------------------------------------- 通用算子

def _grp_corr_nan(m, x, y, min_n=10):
    """组内 pearson; 样本<min_n 或零方差 -> NaN (R45 系约定)。"""
    t = m[["instrument", "td", x, y]].dropna(subset=[x, y])
    t = t.assign(xx=t[x] * t[x], yy=t[y] * t[y], xy=t[x] * t[y])
    gb = t.groupby(["instrument", "td"], sort=False)
    s = gb[[x, y, "xx", "yy", "xy"]].sum()
    n = gb.size()
    num = n * s["xy"] - s[x] * s[y]
    den = np.sqrt((n * s["xx"] - s[x] ** 2) * (n * s["yy"] - s[y] ** 2))
    return (num / den.replace(0, np.nan)).where(n >= min_n)


def _grp_corr_zero(m, x, y, min_n=30):
    """组内 pearson; n<min_n 或零方差 -> 0.0 (R17/R20 系约定)。"""
    t = m[["instrument", "td", x, y]].dropna(subset=[x, y]).copy()
    t["xy"] = t[x] * t[y]
    t["xx"] = t[x] ** 2
    t["yy"] = t[y] ** 2
    a = t.groupby(["instrument", "td"], sort=False).agg(
        n=(x, "size"), sx=(x, "sum"), sy=(y, "sum"),
        sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = a["sxy"] - a["sx"] * a["sy"] / a["n"]
    vx = a["sxx"] - a["sx"] ** 2 / a["n"]
    vy = a["syy"] - a["sy"] ** 2 / a["n"]
    out = cov / np.sqrt(vx.clip(lower=0) * vy.clip(lower=0)).replace(0, np.nan)
    out[(a["n"] < min_n) | (vx <= 0) | (vy <= 0)] = 0.0
    return out.fillna(0.0)


def _winsorize_mad(s, n=3.0):
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0 or mad != mad:
        return s
    return s.clip(med - n * 1.4826 * mad, med + n * 1.4826 * mad)


def _xs_z(df, col):
    """逐日 MAD-winsorize + z (mine_r26_tilt2 口径)。"""
    def _proc(g):
        g = g.copy()
        g[col] = _winsorize_mad(g[col])
        std = g[col].std()
        if std and std > 0:
            g[col] = (g[col] - g[col].mean()) / std
        return g
    out = df.dropna(subset=[col]).replace([np.inf, -np.inf], np.nan).dropna(subset=[col])
    return out.groupby("date", group_keys=False).apply(_proc)


def _to_nm(df, n):
    """1m -> %n 采样, 加 ret/absret/dn/deal_size/volume/amount (R45/R46 口径)。"""
    m = df[df["date"].dt.minute % n == 0].copy()
    gk = ["instrument", "td"]
    prev_c = m.groupby(gk, sort=False)["close"].shift(1)
    m["ret"] = (m["close"] / prev_c - 1.0).fillna(0.0)
    m["absret"] = m["ret"].abs()
    m["volume"] = m["vol_min"]
    m["amount"] = m["amt_min"]
    dn = m["dn_min"].astype(float)
    m["dn"] = dn.where(dn > 0)
    m["deal_size"] = (m["volume"] / dn.replace(0, np.nan)).where(dn > 0)
    return m


def _updn_asym(m):
    """涨/跌 bar 均笔(或均量) 不对称 (ret=0 剔除), 列名 dn 或 volume 自适应。"""
    gk = ["instrument", "td"]
    up = m["ret"] > 0
    dwn = m["ret"] < 0
    vu = m.assign(vu_up=m["_x"] * up, cnt_up=up.astype(float),
                  vu_dn=m["_x"] * dwn, cnt_dn=dwn.astype(float))
    va = vu.groupby(gk, sort=False).agg(
        vu_up=("vu_up", "sum"), cnt_up=("cnt_up", "sum"),
        vu_dn=("vu_dn", "sum"), cnt_dn=("cnt_dn", "sum"))
    mu = va["vu_up"] / va["cnt_up"].replace(0, np.nan)
    md = va["vu_dn"] / va["cnt_dn"].replace(0, np.nan)
    return ((mu - md) / (mu + md).replace(0, np.nan))


def _roll5(s, mp=3):
    return s.rolling(5, min_periods=mp).mean()


# ---------------------------------------------------------------- 特征计算

def _features(df1m, fl, fin, stk_grid):
    """返回 dict[name] -> [date, instrument, factor] (与缓存一致的带符号值)。"""
    F = {}
    m5 = _to_nm(df1m, 5)
    m15 = _to_nm(df1m, 15)
    m30 = _to_nm(df1m, 30)

    def day(s, name):
        d = s.rename(name).reset_index()
        d.columns = ["instrument", "date", name] if d.shape[1] == 3 else d.columns
        return d

    # ---- L1 deal_number 系 ----
    F["deals_absret_corr"] = -_grp_corr_nan(m5, "dn", "absret", 10)
    m5["dn_lag"] = m5.groupby(["instrument", "td"], sort=False)["dn"].shift(1)
    F["deals_ac1"] = -_grp_corr_nan(m5.dropna(subset=["dn", "dn_lag"]),
                                    "dn", "dn_lag", 10)
    q = m5.groupby(["instrument", "td"], sort=False)["deal_size"].agg(["max", "median"])
    F["bigdeal_ratio"] = q["max"] / q["median"].replace(0, np.nan)

    m15["_x"] = m15["dn"]
    F["updn_asym_15m"] = -_updn_asym(m15.dropna(subset=["dn"]))
    m5["_x"] = m5["dn"]
    a5 = _updn_asym(m5.dropna(subset=["dn"])).rename("v").reset_index()
    a5 = a5.sort_values(["instrument", "td"])
    a5["v"] = a5.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["updn_asym_ma5"] = -a5.set_index(["instrument", "td"])["v"]

    # ret_tail3 (1m)
    g1 = ["instrument", "td"]
    df1m["seq"] = df1m.groupby(g1, sort=False).cumcount()
    df1m["nbar"] = df1m.groupby(g1, sort=False)["seq"].transform("max") + 1
    df1m["close_l3"] = df1m.groupby(g1, sort=False)["close"].shift(3)
    last = df1m[df1m["seq"] == df1m["nbar"] - 1]
    F["ret_tail3"] = -(last["close"] / last["close_l3"] - 1.0) \
        .set_axis(pd.MultiIndex.from_frame(last[["instrument", "td"]]))

    # ---- L2 在榜基底 ----
    # z_retrange30_volac1: (z_mad(retrange30)+z_mad(vol_ac1_30m))/2, sign -1
    a = m30.groupby(["instrument", "td"], sort=False).agg(
        rmax=("ret", "max"), rmin=("ret", "min"))
    rr = (a["rmax"] - a["rmin"]).rename("v").reset_index() \
        .rename(columns={"td": "date"})
    m30["vol_l1"] = m30.groupby(g1, sort=False)["volume"].shift(1)
    t = m30.dropna(subset=["vol_l1"]).copy()
    t["xy"] = t["volume"] * t["vol_l1"]
    t["xx"] = t["volume"] ** 2
    t["yy"] = t["vol_l1"] ** 2
    a2 = t.groupby(g1, sort=False).agg(
        n=("volume", "size"), sx=("volume", "sum"), sy=("vol_l1", "sum"),
        sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = a2["sxy"] - a2["sx"] * a2["sy"] / a2["n"]
    vx = a2["sxx"] - a2["sx"] ** 2 / a2["n"]
    vy = a2["syy"] - a2["sy"] ** 2 / a2["n"]
    va = (cov / np.sqrt(vx.clip(lower=0) * vy.clip(lower=0)).replace(0, np.nan))
    va[(a2["n"] < 5) | (vx <= 0) | (vy <= 0)] = np.nan
    va = va.rename("v").reset_index().rename(columns={"td": "date"})
    za = _xs_z(rr, "v").rename(columns={"v": "za"})
    zb = _xs_z(va.dropna(subset=["v"]), "v").rename(columns={"v": "zb"})
    mm = za.merge(zb, on=["date", "instrument"], how="inner")
    F["z_retrange30_volac1"] = -((mm["za"] + mm["zb"]) / 2.0) \
        .set_axis(pd.MultiIndex.from_frame(mm[["date", "instrument"]]))

    # ret_max_15m (raw, 也供 tilt 用)
    rmax15 = m15.groupby(["instrument", "td"], sort=False)["ret"].max()
    F["ret_max_15m"] = rmax15

    # fin cashq / cffps (r13 PIT 口径, ffill 到池网格)
    fin_z = {}
    if fin is not None and len(fin):
        grid = stk_grid[["date", "instrument"]].drop_duplicates()

        def _fin_ratio(num, den):
            a = fin[fin["category"] == "ttm"][["date", "instrument", num]] \
                .rename(columns={num: "vn"})
            b = fin[fin["category"] == "ttm"][["date", "instrument", den]] \
                .rename(columns={den: "vd"})
            m = a.merge(b, on=["date", "instrument"])
            m["fv"] = m["vn"] / (m["vd"] + 1e-8)
            m = m[["date", "instrument", "fv"]].dropna()
            g = grid.merge(m, on=["date", "instrument"], how="left")
            g["fv"] = g.groupby("instrument", group_keys=False)["fv"].ffill()
            return g.dropna(subset=["fv"])

        def _fin_ps(num):
            a = fin[fin["category"] == "ttm"][["date", "instrument", num]] \
                .rename(columns={num: "vn"})
            b = fin[fin["category"] == "lf"][["date", "instrument", "latest_shares"]] \
                .rename(columns={"latest_shares": "sh"})
            m = a.merge(b, on=["date", "instrument"])
            m["fv"] = m["vn"] / (m["sh"] + 1e-8)
            m = m[["date", "instrument", "fv"]].dropna()
            g = grid.merge(m, on=["date", "instrument"], how="left")
            g["fv"] = g.groupby("instrument", group_keys=False)["fv"].ffill()
            return g.dropna(subset=["fv"])

        fin_z["cashq"] = _fin_ratio("net_cffoa", "net_profit")
        fin_z["cffps"] = _fin_ps("net_cfffa")

    rm = rmax15.rename("v").reset_index().rename(columns={"td": "date"})
    zrm = _xs_z(rm, "v").rename(columns={"v": "zb"})
    for fid, fin_name, w, fsign in (("retmax15_cashq", "cashq", 1.0, 1),
                                    ("retmax15_cffps", "cffps", 1.0, -1)):
        zf = _xs_z(fin_z[fin_name], "fv").rename(columns={"fv": "zf"})
        mm2 = zrm.merge(zf, on=["date", "instrument"], how="inner")
        # cache = -[zb * (1+w*(fsign*zf).clip(-2,2))] (cffps FIN_SIGN=-1, 见 r37 #05)
        F[fid] = -(mm2["zb"] * (1.0 + w * (fsign * mm2["zf"]).clip(-2.0, 2.0))) \
            .set_axis(pd.MultiIndex.from_frame(mm2[["date", "instrument"]]))

    # vwapdevchg5_cffps: -z(vwap_dev_chg5) * (1+0.5*clip(-z(cffps),-2,2)), sign +1
    am5 = m5.groupby(["instrument", "td"], sort=False).agg(
        amt=("amount", "sum"), vol=("volume", "sum"), close_d=("close", "last"))
    dev = (am5["close_d"] / (am5["amt"] / am5["vol"].replace(0, np.nan)) - 1.0) \
        .rename("v").reset_index().rename(columns={"td": "date"})
    dev = dev.sort_values(["instrument", "date"])
    dev["v"] = dev["v"] - dev.groupby("instrument", group_keys=False)["v"].shift(5)
    F["vwap_dev_chg5"] = -dev.set_index(["date", "instrument"])["v"]  # sign -1
    zdev = _xs_z(dev.dropna(subset=["v"]), "v").rename(columns={"v": "zb"})
    zcf = _xs_z(fin_z["cffps"], "fv").rename(columns={"fv": "zf"})
    mm3 = zdev.merge(zcf, on=["date", "instrument"], how="inner")
    F["vwapdevchg5_cffps"] = ((-mm3["zb"])
                              * (1.0 + 0.5 * (-mm3["zf"]).clip(-2.0, 2.0))) \
        .set_axis(pd.MultiIndex.from_frame(mm3[["date", "instrument"]]))

    # ---- L3 波动族 ----
    # vwap_disp (raw, R54 口径)
    hl = m5.groupby(["instrument", "td"], sort=False).agg(
        vol=("volume", "sum"), amt=("amount", "sum"))
    vwap_d = (hl["amt"] / hl["vol"].replace(0, np.nan)).rename("vwap")
    m5j = m5.join(vwap_d, on=["instrument", "td"])
    dv = m5j["close"] - m5j["vwap"]
    m5j["wv2"] = m5j["volume"] * dv ** 2
    w2 = m5j.groupby(["instrument", "td"], sort=False).agg(
        wv2=("wv2", "sum"), vol=("volume", "sum"))
    w2 = w2.join(vwap_d, on=["instrument", "td"])
    F["vwap_disp"] = (np.sqrt(w2["wv2"] / w2["vol"].replace(0, np.nan))
                      / w2["vwap"].replace(0, np.nan))

    # imb_x_ret_std (sign -1, R47 口径)
    bv = m5["bid_volume1"] + m5["bid_volume2"] + m5["bid_volume3"]
    av = m5["ask_volume1"] + m5["ask_volume2"] + m5["ask_volume3"]
    ok_vol = (bv + av) > 0
    m5["imb3"] = ((bv - av) / (bv + av)).where(ok_vol)
    m5["imb_x_ret"] = m5["imb3"] * m5["ret"]
    F["imb_x_ret_std"] = -m5.groupby(["instrument", "td"], sort=False)[
        "imb_x_ret"].std()

    # exec_int (raw, R52 口径)
    bno = m5["bid_num_orders1"] + m5["bid_num_orders2"] + m5["bid_num_orders3"]
    ano = m5["ask_num_orders1"] + m5["ask_num_orders2"] + m5["ask_num_orders3"]
    no_total = (bno + ano).where((bno + ano) > 0)
    m5["exec_int"] = (m5["dn_min"].astype(float) / no_total).where(no_total > 0)
    F["exec_int_mean"] = m5.groupby(["instrument", "td"], sort=False)["exec_int"].mean()
    F["exec_int_std"] = m5.groupby(["instrument", "td"], sort=False)["exec_int"].std()

    # vollead_corr_ma5 (sign -1): corr(vol_t, |ret_t+1|) 5m, roll5
    m5["absret_lead"] = m5.groupby(["instrument", "td"], sort=False)["absret"].shift(-1)
    vl = _grp_corr_nan(m5, "volume", "absret_lead", 10).rename("v").reset_index()
    vl = vl.sort_values(["instrument", "td"])
    vl["v"] = vl.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["vollead_corr_ma5"] = -vl.set_index(["instrument", "td"])["v"]

    # turn (raw factorlib)
    F["turn"] = fl.set_index(["date", "instrument"])["turn"]

    # resid_rv5_20 (sign -1, R52 口径; factorlib daily_return)
    P = fl.sort_values(["instrument", "date"]).reset_index(drop=True)
    mkt = P.groupby("date")["daily_return"].mean().rename("mkt")
    P = P.merge(mkt, on="date", how="left")
    g = P.groupby("instrument", group_keys=False)
    xy = P["daily_return"] * P["mkt"]
    y2 = P["mkt"] ** 2
    ex = g["daily_return"].transform(lambda s: s.rolling(60, min_periods=45).mean())
    ey = g["mkt"].transform(lambda s: s.rolling(60, min_periods=45).mean())
    exy = xy.groupby(P["instrument"], group_keys=False).transform(
        lambda s: s.rolling(60, min_periods=45).mean())
    ey2 = y2.groupby(P["instrument"], group_keys=False).transform(
        lambda s: s.rolling(60, min_periods=45).mean())
    beta60 = (exy - ex * ey) / (ey2 - ey ** 2).replace(0, np.nan)
    resid = P["daily_return"] - beta60 * P["mkt"]
    gr = resid.groupby(P["instrument"], group_keys=False)
    rv5 = gr.transform(lambda s: s.rolling(5, min_periods=4).std())
    rv20 = gr.transform(lambda s: s.rolling(20, min_periods=15).std())
    P["resid_rv5_20"] = -(rv5 / rv20.replace(0, np.nan))
    F["resid_rv5_20"] = P.set_index(["date", "instrument"])["resid_rv5_20"]

    # vol_ts_5_20 / rv5_tsz20 (raw, factorlib daily_return)
    rv5r = g["daily_return"].transform(lambda s: s.rolling(5, min_periods=4).std())
    rv20r = g["daily_return"].transform(lambda s: s.rolling(20, min_periods=15).std())
    P["vol_ts_5_20"] = rv5r / rv20r.replace(0, np.nan)
    g5 = rv5r.groupby(P["instrument"], group_keys=False)
    ma20 = g5.transform(lambda s: s.rolling(20, min_periods=15).mean())
    sd20 = g5.transform(lambda s: s.rolling(20, min_periods=15).std())
    P["rv5_tsz20"] = (rv5r - ma20) / sd20.replace(0, np.nan)
    F["vol_ts_5_20"] = P.set_index(["date", "instrument"])["vol_ts_5_20"]
    F["rv5_tsz20"] = P.set_index(["date", "instrument"])["rv5_tsz20"]

    # ---- L4 历史强因子 ----
    # rv_skew (r2 矩偏度, r19b 口径) ma5, 5m/15m
    for m, name in ((m5, "rv_skew_ma5"), (m15, "rv_skew_ma5_15m")):
        m["r2"] = m["ret"] ** 2
        m["r4"] = m["r2"] ** 2
        m["r6"] = m["r2"] ** 3
        a = m.groupby(["instrument", "td"], sort=False).agg(
            n=("r2", "size"), s2=("r2", "sum"), s4=("r4", "sum"), s6=("r6", "sum"))
        m1, m2_, m3 = a["s2"] / a["n"], a["s4"] / a["n"], a["s6"] / a["n"]
        var = (m2_ - m1 ** 2).clip(lower=0)
        sk = ((m3 - 3 * m1 * m2_ + 2 * m1 ** 3) / var.pow(1.5).replace(0, np.nan)) \
            .rename("v").reset_index()
        sk = sk.sort_values(["instrument", "td"])
        sk["v"] = sk.groupby("instrument", group_keys=False)["v"].transform(_roll5)
        F[name] = -sk.set_index(["instrument", "td"])["v"]  # sign -1

    # upvol_asym_ma5 / upvol_asym_ma5_15m (volume 版, sign -1 缓存为 -raw)
    for m, name in ((m5, "upvol_asym_ma5"), (m15, "upvol_asym_ma5_15m")):
        m["_x"] = m["volume"]
        ua = _updn_asym(m).rename("v").reset_index()
        ua = ua.sort_values(["instrument", "td"])
        ua["v"] = ua.groupby("instrument", group_keys=False)["v"].transform(_roll5)
        F[name] = -ua.set_index(["instrument", "td"])["v"]

    # kurt_ma5 (r19c 口径: 5m ret 超额峰度, 全 bar; roll5; sign -1)
    m5["r1"] = m5["ret"]
    m5["s2_"] = m5["r1"] ** 2
    m5["s3_"] = m5["r1"] ** 3
    m5["s4_"] = m5["r1"] ** 4
    a = m5.groupby(["instrument", "td"], sort=False).agg(
        n=("r1", "size"), s1=("r1", "sum"), s2=("s2_", "sum"),
        s3=("s3_", "sum"), s4=("s4_", "sum"))
    n = a["n"]
    m1 = a["s1"] / n
    m2v = a["s2"] / n - m1 ** 2
    m4 = (a["s4"] / n - 4 * m1 * a["s3"] / n
          + 6 * m1 ** 2 * a["s2"] / n - 3 * m1 ** 4)
    kd = (m4 / m2v.clip(lower=0) ** 2 - 3.0)
    kd[m2v <= 0] = np.nan
    kd = kd.rename("v").reset_index()
    kd = kd.sort_values(["instrument", "td"])
    kd["v"] = kd.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["kurt_ma5"] = -kd.set_index(["instrument", "td"])["v"]  # sign -1

    # n_reversals_30m (B_N_REV_30M, sign -1)
    s = np.sign(m30["ret"]).replace(0, np.nan)
    sff = s.groupby([m30["instrument"], m30["td"]], sort=False).ffill()
    prev_s = sff.groupby([m30["instrument"], m30["td"]], sort=False).shift(1)
    rev = ((sff * prev_s) < 0).astype(float)
    F["n_reversals_30m"] = -rev.groupby([m30["instrument"], m30["td"]],
                                        sort=False).sum()

    # absret_ac1_ma5 (R17 系 fill-0 corr min30, roll5, sign -1)
    m5["absret_l1"] = m5.groupby(["instrument", "td"], sort=False)["absret"].shift(1)
    ac = _grp_corr_zero(m5.dropna(subset=["absret_l1"]), "absret", "absret_l1", 30) \
        .rename("v").reset_index()
    ac = ac.sort_values(["instrument", "td"])
    ac["v"] = ac.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["absret_ac1_ma5"] = -ac.set_index(["instrument", "td"])["v"]  # sign -1

    # gap_freq_20 (sign -1): |open/pre_close-1|>0.01 的 20 日均 (min10)
    # down_vol_share_15 (sign +1)
    # park5 / gk5 (raw)
    dy = df1m.groupby(["instrument", "td"], sort=False).agg(
        open=("open", "first"), high=("high", "max"), low=("low", "min"),
        close=("close", "last"))
    dy = dy.reset_index().rename(columns={"td": "date"})
    dy = dy.sort_values(["instrument", "date"]).reset_index(drop=True)
    dy["pre_close"] = dy.groupby("instrument", group_keys=False)["close"].shift(1)
    # 注: 平台 pre_close 官方昨收与 shift(close) 在非除权日一致;
    # r17 系缓存经实测与 shift(close) 口径最接近 (gap 0.99995/down_vol 0.99998,
    # 优于 csv pre_close 比例修正的 0.993/0.977), 故用 shift 口径。
    dy["gap"] = dy["open"] / dy["pre_close"] - 1.0
    is_gap = (dy["gap"].abs() > 0.01).astype(float).where(dy["gap"].notna())
    dy["ret"] = dy["close"] / dy["pre_close"] - 1.0
    dy["ret_dn2"] = np.minimum(dy["ret"], 0.0) ** 2
    dy["ret2"] = dy["ret"] ** 2
    gdy = dy.groupby("instrument", group_keys=False)
    F["gap_freq_20"] = -is_gap.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(20, min_periods=10).mean()) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    dn15 = gdy["ret_dn2"].transform(lambda s: s.rolling(15, min_periods=7).sum())
    tt15 = gdy["ret2"].transform(lambda s: s.rolling(15, min_periods=7).sum())
    F["down_vol_share_15"] = (dn15 / tt15.replace(0, np.nan)) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    ln_hl2 = np.log(dy["high"] / dy["low"]) ** 2
    ln_co2 = np.log(dy["close"] / dy["open"]) ** 2
    m5_hl = ln_hl2.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(5, min_periods=4).mean())
    m5_co = ln_co2.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(5, min_periods=4).mean())
    F["park5"] = np.sqrt(m5_hl / (4.0 * np.log(2))) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    F["gk5"] = np.sqrt((m5_hl * 0.5 - m5_co * (2 * np.log(2) - 1)).clip(lower=0)) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))

    # b_pvsign_resid_rank_top1 (sign -1; 池内残差, R20 notebook 口径)
    prev_c1 = df1m.groupby(g1, sort=False)["close"].shift(1)
    df1m["ret1"] = df1m["close"] / prev_c1 - 1.0
    ok = df1m["ret1"].notna() & (df1m["vol_min"] >= 0)
    t1 = df1m[ok].copy()
    t1["sv"] = np.sign(t1["ret1"]) * t1["vol_min"]
    pr = t1.groupby(g1, sort=False).agg(sv=("sv", "sum"), vv=("vol_min", "sum"))
    pr["pressure"] = -(pr["sv"] / (pr["vv"] + 1e-8))
    prev_v5 = m5.groupby(g1, sort=False)["volume"].shift(1)
    m5["dv"] = (m5["volume"] / prev_v5.replace(0, np.nan) - 1.0).fillna(0.0)
    pvs = _grp_corr_zero(m5, "dv", "ret", 30).rename("pvsign").reset_index()
    day = pvs.merge(pr[["pressure"]].reset_index(), on=g1, how="inner")
    day = day.rename(columns={"td": "date"})
    day = day.merge(stk_grid[["date", "instrument"]].drop_duplicates(),
                    on=["date", "instrument"], how="inner")

    def _resid(g):
        y = g["pvsign"].to_numpy(dtype=float)
        x = g["pressure"].rank().to_numpy(dtype=float)
        okk = ~np.isnan(y) & ~np.isnan(x)
        res = np.full_like(y, np.nan)
        if okk.sum() >= 31:
            A = np.column_stack([np.ones(okk.sum()), x[okk]])
            beta, *_ = np.linalg.lstsq(A, y[okk], rcond=None)
            res[okk] = y[okk] - A @ beta
        return pd.Series(res, index=g.index)

    day["pvsign_resid"] = day.groupby("date", group_keys=False).apply(_resid)
    F["b_pvsign_resid_rank_top1"] = -day.set_index(["date", "instrument"])[
        "pvsign_resid"]

    # nm_vol_ret_corr / nm_n_reversals (【5m】口径, recompute_nm3.factors_from_5m:
    # corr(5m volume, |5m ret|) 与 5m close.diff 非零符号变号次数, n<30->0.0, sign -1)
    F["nm_vol_ret_corr"] = -_grp_corr_zero(m5, "volume", "absret", 30)
    s1 = np.sign((m5["close"] - m5.groupby(g1, sort=False)["close"].shift(1))
                 .fillna(0.0))
    regime = s1.replace(0, np.nan).groupby([m5["instrument"], m5["td"]],
                                           sort=False).ffill()
    prev_regime = regime.groupby([m5["instrument"], m5["td"]],
                                 sort=False).shift(1)
    nrev = (regime.notna() & prev_regime.notna()
            & (regime != prev_regime)).astype(float)
    nrev_d = nrev.groupby([m5["instrument"], m5["td"]], sort=False).sum()
    nbar5 = m5.groupby(g1, sort=False)["close"].size()
    F["nm_n_reversals"] = -nrev_d.where(nbar5 >= 30, 0.0)


    # ================= R62 扩展特征 (raw 无 sign, 与挖掘缓存一致) =================
    # ---- 盘口动态 (R47 bookdyn 口径, 5m) ----
    mid = (m5["bid_price1"] + m5["ask_price1"]) / 2
    ok_book = (m5["ask_price1"] > m5["bid_price1"]) & (m5["bid_price1"] > 0) \
        & (mid > 0)
    m5["spread"] = ((m5["ask_price1"] - m5["bid_price1"]) / mid).where(ok_book)
    bv = m5["bid_volume1"] + m5["bid_volume2"] + m5["bid_volume3"]
    av = m5["ask_volume1"] + m5["ask_volume2"] + m5["ask_volume3"]
    ok_vol = (bv + av) > 0
    m5["imb1"] = ((m5["bid_volume1"] - m5["ask_volume1"])
                  / (m5["bid_volume1"] + m5["ask_volume1"]).replace(0, np.nan))
    m5["imb3"] = ((bv - av) / (bv + av)).where(ok_vol)
    m5["depth"] = np.log((bv + av).where(ok_vol))
    bno = m5["bid_num_orders1"] + m5["bid_num_orders2"] + m5["bid_num_orders3"]
    ano = m5["ask_num_orders1"] + m5["ask_num_orders2"] + m5["ask_num_orders3"]
    m5["ono"] = (bno + ano).where((bno + ano) > 0)
    _bs = bv / bno.replace(0, np.nan)
    _as = av / ano.replace(0, np.nan)
    m5["osize_asym"] = ((_bs - _as) / (_bs + _as)).where((_bs > 0) & (_as > 0))
    m5["slope_asym"] = (
        ((m5["bid_volume1"] - m5["bid_volume3"])
         / (m5["bid_volume1"] + m5["bid_volume3"]).replace(0, np.nan))
        - ((m5["ask_volume1"] - m5["ask_volume3"])
           / (m5["ask_volume1"] + m5["ask_volume3"]).replace(0, np.nan)))
    m5["book_slope"] = (((m5["bid_price1"] - m5["bid_price3"])
                         + (m5["ask_price3"] - m5["ask_price1"]))
                        / mid).where(ok_book)
    mp_den = (m5["bid_volume1"] + m5["ask_volume1"]).replace(0, np.nan)
    _mp = (m5["bid_price1"] * m5["ask_volume1"]
           + m5["ask_price1"] * m5["bid_volume1"]) / mp_den
    m5["mp_dev"] = (_mp - m5["close"]) / m5["close"].replace(0, np.nan)
    m5["imb_x_ret"] = m5["imb3"] * m5["ret"]
    bk = m5.groupby(g1, sort=False).agg(
        depth_mean=("depth", "mean"), depth_std=("depth", "std"),
        depth_skew=("depth", "skew"),
        book_slope_mean=("book_slope", "mean"),
        book_slope_std=("book_slope", "std"),
        slope_asym_std=("slope_asym", "std"),
        osize_asym_mean=("osize_asym", "mean"),
        osize_asym_std=("osize_asym", "std"),
        imb3_std=("imb3", "std"), imb1_std=("imb1", "std"),
        imb1_skew=("imb1", "skew"),
        imb_x_ret_mean=("imb_x_ret", "mean"),
        mp_dev_mean=("mp_dev", "mean"), mp_dev_std=("mp_dev", "std"),
        spread_mean=("spread", "mean"), spread_std=("spread", "std"))
    for _c in bk.columns:
        F[_c] = bk[_c]
    m5["imb3_lag"] = m5.groupby(g1, sort=False)["imb3"].shift(1)
    F["imb3_ac1"] = _grp_corr_nan(m5.dropna(subset=["imb3", "imb3_lag"]),
                                  "imb3", "imb3_lag", 10)
    m5["mp_dev_lag"] = m5.groupby(g1, sort=False)["mp_dev"].shift(1)
    F["mp_dev_ac1"] = _grp_corr_nan(m5.dropna(subset=["mp_dev", "mp_dev_lag"]),
                                    "mp_dev", "mp_dev_lag", 10)
    F["orders_ret_corr"] = _grp_corr_nan(m5, "ono", "absret", 10)
    F["osize_ret_corr"] = _grp_corr_nan(m5, "osize_asym", "ret", 10)
    _am = (m5["mins"] >= MORNING[0]) & (m5["mins"] <= MORNING[1])
    _amb = m5[_am].groupby(g1, sort=False).agg(
        spread_am=("spread", "mean"), imb3_am=("imb3", "mean"),
        depth_am=("depth", "mean"))
    for _c in _amb.columns:
        F[_c] = _amb[_c]

    # ---- lead-lag + 市场联动 + 时间重心 (R55 口径, 5m) ----
    m5["absret_lead"] = m5.groupby(g1, sort=False)["absret"].shift(-1)
    m5["vol_lead"] = m5.groupby(g1, sort=False)["volume"].shift(-1)
    m5["dn_lead"] = m5.groupby(g1, sort=False)["dn"].shift(-1)
    F["vollead_corr"] = _grp_corr_nan(m5, "volume", "absret_lead", 10)
    F["deallead_corr"] = _grp_corr_nan(m5, "dn", "absret_lead", 10)
    F["retlead_corr"] = _grp_corr_nan(m5, "ret", "vol_lead", 10)
    F["absretlead_corr"] = _grp_corr_nan(m5, "absret", "dn_lead", 10)
    _mkt = m5.groupby(["td", "mins"], sort=False)["ret"].median().rename("mkt_ret")
    m5 = m5.join(_mkt, on=["td", "mins"])
    F["mkt_corr"] = _grp_corr_nan(m5, "ret", "mkt_ret", 10)
    F["mkt_corr_am"] = _grp_corr_nan(m5[_am], "ret", "mkt_ret", 5)
    m5["amt_x_mins"] = m5["amount"] * m5["mins"]
    m5["dn_x_mins"] = m5["dn"] * m5["mins"]
    _c = m5.groupby(g1, sort=False).agg(
        amt_sum=("amount", "sum"), amt_x=("amt_x_mins", "sum"),
        dn_sum=("dn", "sum"), dn_x=("dn_x_mins", "sum"))
    F["amt_center"] = _c["amt_x"] / _c["amt_sum"].replace(0, np.nan)
    F["deal_center"] = _c["dn_x"] / _c["dn_sum"].replace(0, np.nan)

    # ---- deal 15m (R46 口径) ----
    d15 = m15.groupby(g1, sort=False).agg(
        deals_total=("dn", "sum"), deals_mean=("dn", "mean"),
        deals_std=("dn", "std"), deals_skew=("dn", "skew"))
    d15["deals_cv"] = d15["deals_std"] / d15["deals_mean"].replace(0, np.nan)
    _sub = m15.dropna(subset=["dn"])
    _up = _sub["ret"] > 0
    _dn = _sub["ret"] < 0
    _va = _sub.assign(dn_up=_sub["dn"] * _up, cnt_up=_up.astype(float),
                      dn_dn=_sub["dn"] * _dn, cnt_dn=_dn.astype(float)) \
        .groupby(g1, sort=False).agg(
            dn_up=("dn_up", "sum"), cnt_up=("cnt_up", "sum"),
            dn_dn=("dn_dn", "sum"), cnt_dn=("cnt_dn", "sum"))
    _mu = _va["dn_up"] / _va["cnt_up"].replace(0, np.nan)
    _md = _va["dn_dn"] / _va["cnt_dn"].replace(0, np.nan)
    d15["deals_updn_asym"] = (_mu - _md) / (_mu + _md).replace(0, np.nan)
    for _c in d15.columns:
        F[_c] = d15[_c]

    # ---- exec 补充 (R52 口径, 5m; dn>0 过滤版 exec_int) ----
    m5["no_total"] = no_total
    _exi = (m5["dn"] / no_total.replace(0, np.nan)).where(no_total > 0)
    _eb = m5["dn"] / bno.replace(0, np.nan)
    _ea = m5["dn"] / ano.replace(0, np.nan)
    m5["fill_asym"] = ((_eb - _ea) / (_eb + _ea)).where((_eb > 0) & (_ea > 0))
    m5["exec_sgn"] = _exi * np.sign(m5["ret"])
    _ex = m5.groupby(g1, sort=False).agg(
        fill_asym_mean=("fill_asym", "mean"),
        exec_sgn_sum=("exec_sgn", "sum"), n_bars=("ret", "size"))
    F["fill_asym_mean"] = _ex["fill_asym_mean"]
    F["exec_x_ret"] = _ex["exec_sgn_sum"] / _ex["n_bars"].replace(0, np.nan)
    F["deals_orders_corr"] = _grp_corr_nan(m5, "dn", "no_total", 10)

    # ---- 尾盘竞价补充 (R52 口径, 1m; ret_tail3 已有) ----
    df1m["close_l1"] = df1m.groupby(g1, sort=False)["close"].shift(1)
    df1m["seq"] = df1m.groupby(g1, sort=False).cumcount()
    df1m["nbar"] = df1m.groupby(g1, sort=False)["seq"].transform("max") + 1
    _tot = df1m.groupby(g1, sort=False).agg(
        dn_total=("dn_min", "sum"), vol_total=("vol_min", "sum"))
    _last = df1m[df1m["seq"] == df1m["nbar"] - 1].set_index(g1)
    _t3 = df1m[df1m["seq"] >= df1m["nbar"] - 3] \
        .groupby(g1, sort=False)["dn_min"].sum()
    _dn1 = _last["dn_min"].astype(float)
    F["deals_tail1_share"] = (_dn1 / _tot["dn_total"].replace(0, np.nan))
    F["vol_tail1_share"] = (_last["vol_min"]
                            / _tot["vol_total"].replace(0, np.nan))
    F["ret_tail1"] = (_last["close"] / _last["close_l1"] - 1.0).fillna(0.0)
    F["tail1_deal_size"] = (_last["vol_min"]
                            / _dn1.replace(0, np.nan)).where(_dn1 > 0)
    F["tail3_deals_share"] = (_t3 / _tot["dn_total"].replace(0, np.nan))

    # ---- 量价价格分布 (R54 口径, 5m; vwap_disp 已有) ----
    _hl = m5.groupby(g1, sort=False)["close"].agg(["max", "min"])
    m5 = m5.join(_hl.rename(columns={"max": "h", "min": "l"}), on=g1)
    _rng = (m5["h"] - m5["l"]).replace(0, np.nan)
    _top = (m5["close"] >= m5["l"] + 2.0 / 3.0 * _rng).astype(float) \
        .where(_rng.notna())
    _bot = (m5["close"] <= m5["l"] + 1.0 / 3.0 * _rng).astype(float) \
        .where(_rng.notna())
    m5["vol_top"] = m5["volume"] * _top
    m5["vol_bot"] = m5["volume"] * _bot
    _a = m5.groupby(g1, sort=False).agg(
        vol=("volume", "sum"), amt=("amount", "sum"),
        vol_top=("vol_top", "sum"), vol_bot=("vol_bot", "sum"))
    F["vol_top_third"] = _a["vol_top"] / _a["vol"].replace(0, np.nan)
    F["vol_bot_third"] = _a["vol_bot"] / _a["vol"].replace(0, np.nan)
    _vwap = (_a["amt"] / _a["vol"].replace(0, np.nan)).rename("vwap")
    m5 = m5.join(_vwap, on=g1)
    _dev = m5["close"] - m5["vwap"]
    m5["wv2"] = m5["volume"] * _dev ** 2
    m5["wv3"] = m5["volume"] * _dev ** 3
    _m = m5.groupby(g1, sort=False).agg(wv2=("wv2", "sum"), wv3=("wv3", "sum"),
                                        vol=("volume", "sum"))
    _m["vw_std"] = np.sqrt(_m["wv2"] / _m["vol"].replace(0, np.nan))
    F["vwap_skew"] = (_m["wv3"] / _m["vol"].replace(0, np.nan)) \
        / _m["vw_std"].replace(0, np.nan) ** 3

    out = {}
    for name, s in F.items():
        d = s.rename("factor").reset_index()
        d = d.rename(columns={"td": "date"})
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()
        out[name] = d[["date", "instrument", "factor"]] \
            .replace([np.inf, -np.inf], np.nan)
    return out


# ---------------------------------------------------------------- 模型推理

def _model_year(y):
    if y <= 2020:
        return 2020
    if y == 2021:
        return 2021
    if y == 2022:
        return 2022
    if y == 2023:
        return 2023
    return 2024


def _predict_json_model_vec(model, values):
    """纯 numpy 向量化 XGBoost JSON 推理 (无 xgboost 时的兜底)。"""
    raw = model["learner"]["learner_model_param"].get("base_score", "0")
    base = float(str(raw).strip("[]"))
    pred = np.full(values.shape[0], base, dtype=float)
    trees = model["learner"]["gradient_booster"]["model"]["trees"]
    for tree in trees:
        left = np.asarray(tree["left_children"])
        right = np.asarray(tree["right_children"])
        split_index = np.asarray(tree["split_indices"])
        split_cond = np.asarray(tree["split_conditions"], dtype=float)
        default_left = np.asarray(tree["default_left"])
        node = np.zeros(values.shape[0], dtype=int)
        active = left[node] != -1
        while active.any():
            idx = np.flatnonzero(active)
            nd = node[idx]
            fv = values[idx, split_index[nd]]
            go_left = np.where(~np.isfinite(fv), default_left[nd] == 1,
                               fv < split_cond[nd])
            node[idx] = np.where(go_left, left[nd], right[nd])
            active = left[node] != -1
        pred += split_cond[node]
    return pred



import base64 as _b64
import tempfile as _tf

MODELS_B64 = {'2020': '...', '2021': '...', '2022': '...', '2023': '...', '2024': '...'}
MODEL_FEATS = ['absret_ac1_ma5', 'absretlead_corr', 'amt_center', 'b_pvsign_resid_rank_top1', 'bigdeal_ratio', 'book_slope_mean', 'book_slope_std', 'deal_center', 'deallead_corr', 'deals_absret_corr', 'deals_ac1', 'deals_cv', 'deals_mean', 'deals_orders_corr', 'deals_skew', 'deals_std', 'deals_tail1_share', 'deals_total', 'deals_updn_asym', 'depth_am', 'depth_mean', 'depth_skew', 'depth_std', 'down_vol_share_15', 'exec_int_mean', 'exec_int_std', 'exec_x_ret', 'fill_asym_mean', 'gap_freq_20', 'gk5', 'imb1_skew', 'imb1_std', 'imb3_ac1', 'imb3_am', 'imb3_std', 'imb_x_ret_mean', 'imb_x_ret_std', 'kurt_ma5', 'mkt_corr', 'mkt_corr_am', 'mp_dev_ac1', 'mp_dev_mean', 'mp_dev_std', 'n_reversals_30m', 'nm_n_reversals', 'nm_vol_ret_corr', 'orders_ret_corr', 'osize_asym_mean', 'osize_asym_std', 'osize_ret_corr', 'park5', 'resid_rv5_20', 'ret_max_15m', 'ret_tail1', 'ret_tail3', 'retlead_corr', 'retmax15_cashq', 'retmax15_cffps', 'rv5_tsz20', 'rv_skew_ma5', 'rv_skew_ma5_15m', 'slope_asym_std', 'spread_am', 'spread_mean', 'spread_std', 'tail1_deal_size', 'tail3_deals_share', 'turn', 'updn_asym_15m', 'updn_asym_ma5', 'upvol_asym_ma5', 'upvol_asym_ma5_15m', 'vol_bot_third', 'vol_tail1_share', 'vol_top_third', 'vol_ts_5_20', 'vollead_corr', 'vollead_corr_ma5', 'vwap_dev_chg5', 'vwap_disp', 'vwap_skew', 'vwapdevchg5_cffps', 'z_retrange30_volac1']


def _predict_year(year, values):
    payload = _b64.b64decode(MODELS_B64[str(_model_year(year))])
    try:
        import xgboost as xgb
        booster = xgb.Booster()
        with _tf.NamedTemporaryFile(suffix=".json", delete=False) as tf:
            tf.write(payload)
            tmp = tf.name
        try:
            booster.load_model(tmp)
        finally:
            os.remove(tmp)
        dm = xgb.DMatrix(values, feature_names=MODEL_FEATS)
        return booster.predict(dm)
    except Exception:
        # 平台 xgboost 版本不一致/无法读 3.1.1 JSON/临时目录受限 时,
        # 用纯 numpy 树遍历 (版本无关, 与 booster 输出已验证 corr=1.0)
        model = json.loads(payload.decode("utf-8"))
        return _predict_json_model_vec(model, values)


# ---------------------------------------------------------------- 主入口

def main(datasources, start_date, end_date):
    import dai  # noqa: F401  (平台环境)
    df1m = _load_bar1m(datasources, start_date, end_date, buf_days=100)
    fl = _load_factorlib(start_date, end_date, buf_days=100)
    fin = _load_fin(datasources, start_date, end_date)
    stk_grid = _stk("2019-01-01", _fin_grid_end(end_date))

    feats = _features(df1m, fl, fin, stk_grid)

    panel = None
    for name in FEATURE_COLS:
        f = feats[name].rename(columns={"factor": name})
        panel = f if panel is None else panel.merge(
            f, on=["date", "instrument"], how="inner")
    panel = panel.sort_values(["date", "instrument"]).reset_index(drop=True)
    panel = panel[(panel["date"] >= pd.to_datetime(start_date))
                  & (panel["date"] <= pd.to_datetime(end_date))]
    # 先收敛到股票池再 z: 训练面板的特征缓存本身就只含池内股票,
    # 若 z 在全市场(含池外 ~2000 只)上算, 均值/std 不同导致输出偏移
    stk = _stk(start_date, end_date)
    panel = panel.merge(stk, on=["date", "instrument"], how="inner")
    for name in FEATURE_COLS:
        g = panel.groupby("date", group_keys=False)[name]
        panel[name] = (panel[name] - g.transform("mean")) \
            / g.transform("std").replace(0, np.nan)

    panel["year"] = panel["date"].dt.year
    pred = np.full(len(panel), np.nan)
    for yr in sorted(panel["year"].unique()):
        idx = np.flatnonzero((panel["year"] == yr).to_numpy())
        values = panel.iloc[idx][MODEL_FEATS].fillna(0.0) \
            .astype("float32").to_numpy()
        pred[idx] = _predict_year(int(yr), values)
    panel["factor"] = pred * SIGN

    out = panel
    out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["factor"])
    return out[["date", "instrument", "factor"]]
